## CUDA Reproduction — Perishable Inventory Value Iteration

### Notebook roadmap
1. CUDA environment and experiment parameters
2. Poisson demand and inventory-capped sales
3. GPU Bellman update for a single product
4. **Experiment A:** Single product, `M=2, Q=9` (GPU)
5. **Experiment B:** Single product, `M=3, Q=15` (GPU)
6. Two-product model with one-way substitution from B to A
7. **Experiment C:** Two products, `Qa=Qb=10` (GPU)
8. Monte Carlo policy evaluation over 400,000 simulated days (CPU)
9. CPU/GPU validation, timing, and interpretation

**Implementation note:** The value-iteration Bellman updates run on an **NVIDIA GPU using Numba CUDA**. Probability tables are prepared on the CPU and transferred once to the GPU. Simulation remains sequential on the CPU because each day's inventory depends on the preceding day.

**Execution:** Use a CUDA-enabled NVIDIA GPU (e.g. Google Colab: Runtime → Change runtime type → T4 GPU, when available). Apple Silicon does not support NVIDIA CUDA. Run cells from top to bottom.


## 0. Environment and dependencies

Install dependencies in a Colab code cell, if necessary:

```python
%pip install numpy scipy numba numba-cuda matplotlib pandas
```

- `numba.cuda`: compiles GPU kernels and launches parallel threads.
- `cuda.to_device(...)`: transfers arrays from CPU memory to GPU memory.
- `cuda.grid(1)`: assigns one index to each GPU thread.
- `cuda.synchronize()`: waits for GPU work to finish before timing.

The code **requires an NVIDIA GPU with a working CUDA driver**. For a small correctness test without a GPU, Numba supports `NUMBA_ENABLE_CUDASIM=1` set **before importing Numba**. The simulator is not suitable for performance benchmarking.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import poisson, binom
from numba import cuda
from IPython.display import display

MU_A = MU_B = 5.0
PRICE_A = PRICE_B = 1.0
COST_A = COST_B = 0.5
GAMMA = 0.5
EPSILON = 1e-4
SEED = 123

if not cuda.is_available():
    raise RuntimeError(
        "CUDA GPU unavailable. In Google Colab, select Runtime > Change runtime type > T4 GPU "
        "and reconnect. NVIDIA CUDA does not run directly on Apple Silicon."
    )

print("NumPy:", np.__version__)
print("CUDA available:", cuda.is_available())
cuda.detect()


## 1. Poisson demand and inventory-capped sales

Daily demand is modeled as \(D\sim\mathrm{Poisson}(\mu)\). With \(y\) units available, realized sales are \(S=\min(D,y)\).

- `poisson.pmf(d, mu)`: probability of exactly \(d\) units of demand.
- `poisson.sf(y-1, mu)`: probability of demand greater than or equal to \(y\).
- `capped_poisson(mu, y)`: probability distribution of feasible sales \(S\in\{0,\ldots,y\}\).

This step **computes a probability distribution analytically**; it does not generate random observations or create a training dataset.


In [ ]:
def capped_poisson(mu, cap):
    if cap==0:return np.array([1.])
    return np.r_[poisson.pmf(np.arange(cap),mu),poisson.sf(cap-1,mu)]

In [ ]:
y = 3
sales_probs = capped_poisson(5.0, y)
display(pd.DataFrame({"sales": range(y+1), "probability": sales_probs}))
print("Probability sum:", sales_probs.sum())
plt.figure(figsize=(5.5,3))
plt.bar(range(y+1), sales_probs)
plt.xlabel("Sales (capped by inventory)"); plt.ylabel("Probability")
plt.title("Poisson demand (mu=5), inventory=3")
plt.show()

## 2. Single-product model: state and GPU Bellman update

For shelf life `M=2`, the state is `(old, fresh)`. Orders `q` arrive on the **following day**, and products are sold **FIFO**.

\[
V_{k+1}(I)=s\,\mathbb E[S\mid I]+\max_{0\le q\le Q}\left[-cq+\sum_{d=0}^Y P(S=d\mid Y)V_k(F(I,q,d))\right].
\]

- `single_update_gpu`: **one GPU thread per inventory state**; loops over feasible order quantities and capped sales to find the best value and order.
- `W` and `V`: separate GPU arrays for the previous and next Bellman iterations (**synchronous update**).
- `increments`: copied to CPU each iteration to compute the **span** and the estimated **average reward**.
- `policy[j]`: optimal order for state `j`.

All expected values are summed analytically; the GPU kernel does **not** simulate demand.


In [ ]:
def solve_single(M=2, Q=9, mu=5., s=1., c=.5, epsilon=EPSILON, max_iter=120):
    # Build states and capped Poisson probabilities once on the CPU.
    states = np.array([np.unravel_index(j, (Q+1,)*M)
                       for j in range((Q+1)**M)], dtype=np.int32)
    ps = np.zeros((Q*M+1, Q*M+1), dtype=np.float64)
    expected = np.zeros(Q*M+1, dtype=np.float64)
    for y in range(Q*M+1):
        p = capped_poisson(mu, y)
        ps[y, :y+1] = p
        expected[y] = np.arange(y+1) @ p

    # Transfer fixed inputs only once; keep value arrays on the GPU.
    d_states = cuda.to_device(states)
    d_ps = cuda.to_device(ps)
    d_expected = cuda.to_device(expected)
    n = len(states)
    d_w = cuda.to_device(np.zeros(n, dtype=np.float64))
    d_v = cuda.device_array(n, dtype=np.float64)
    d_policy = cuda.device_array(n, dtype=np.int32)
    d_inc = cuda.device_array(n, dtype=np.float64)

    threads = 128
    blocks = (n + threads - 1) // threads
    spans, gains = [], []
    cuda.synchronize()
    t0 = time.perf_counter()
    for iteration in range(max_iter):
        single_update_gpu[blocks, threads](d_states, d_ps, d_expected,
                                           d_w, d_v, d_policy, d_inc,
                                           Q, M, s, c)
        # This transfer also synchronizes the completed kernel.
        delta = d_inc.copy_to_host()
        span = float(delta.max() - delta.min())
        spans.append(span)
        gains.append(float(delta.mean()))
        if span < epsilon:
            break
        # Swap buffers; the next iteration reads only the completed V.
        d_w, d_v = d_v, d_w
    cuda.synchronize()
    runtime = time.perf_counter() - t0
    result = (states, d_v.copy_to_host(), d_policy.copy_to_host(),
              np.array(spans), np.array(gains))
    print(f"GPU Bellman runtime (excluding table preparation) = {runtime:.3f}s")
    return result


In [ ]:
@cuda.jit
def single_update_gpu(states, ps, expected, W, V, policy, increments, Q, M, price, cost):
    # Each thread updates one inventory state and searches all possible orders.
    j = cuda.grid(1)
    if j >= states.shape[0]:
        return

    Y = 0
    for r in range(M):
        Y += states[j, r]
    best = -1e100
    best_q = 0

    for q in range(Q + 1):
        future = 0.0
        for d in range(Y + 1):
            # Compute FIFO next-state index directly in base (Q + 1).
            k = 0
            for r in range(M - 1):
                used_from_newer = d
                for t in range(r + 1):
                    used_from_newer -= states[j, t]
                if used_from_newer < 0:
                    used_from_newer = 0
                remaining = states[j, r + 1] - used_from_newer
                if remaining < 0:
                    remaining = 0
                k = k * (Q + 1) + remaining
            k = k * (Q + 1) + q
            future += ps[Y, d] * W[k]
        val = price * expected[Y] - cost * q + future
        if val > best:
            best = val
            best_q = q

    V[j] = best
    policy[j] = best_q
    increments[j] = best - W[j]


### Worked example: state indexing and FIFO transition

For `M=2`, the state `(old, fresh)` is mapped to a one-dimensional index by `old*(Q+1) + fresh`. Starting from `(2,3)`, suppose four units are sold and six units are ordered. FIFO sales leave one fresh unit, which becomes old stock tomorrow; the six ordered units arrive as fresh stock. Thus the next state is **`(1,6)`**.


In [ ]:
Q_demo = 9
old, fresh, sales, order = 2, 3, 4, 6
remaining_fresh = max(0, fresh - max(0, sales - old))
next_state = (remaining_fresh, order)
state_index = old * (Q_demo + 1) + fresh
next_index = next_state[0] * (Q_demo + 1) + next_state[1]
print("Current state:", (old, fresh), "index:", state_index)
print("Sales:", sales, "Order:", order)
print("Next state:", next_state, "index:", next_index)

## 3. Experiment A — Single product (`M=2, Q=9`) [GPU]

Compute the optimal ordering policy over **100 states**. Each GPU thread updates one state by evaluating all possible orders and capped Poisson sales.

**Reference:** average daily profit approximately **2.215**. Store the result as `one_m2` for the plots and Monte Carlo simulation below.


In [ ]:
t0 = time.perf_counter()
states, values, policy, spans, gains = solve_single(M=2, Q=9)
one_m2 = dict(M=2, Q=9, states=states, values=values, policy=policy,
              spans=spans, gains=gains)
print(f"GPU VI daily profit = {gains[-1]:.6f} (paper ~2.215)")
print(f"Iterations = {len(spans)}, final span = {spans[-1]:.7g}")
print(f"Optimal order at zero inventory = {int(policy[0])}")
print(f"Wall time (including setup/compilation) = {time.perf_counter()-t0:.2f}s")


### Convergence diagnostics: span and mean value increment

The stopping rule is \(\mathrm{span}_k<\varepsilon\), where

\[
\mathrm{span}_k=\max_I\left[V_{k+1}(I)-V_k(I)\right]-\min_I\left[V_{k+1}(I)-V_k(I)\right].
\]

The maximum and minimum are taken **across all states within the same iteration**, not across different iterations. A small span indicates that the state-wise Bellman increments have become nearly uniform.


In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,3.6))
iterations = np.arange(1, len(one_m2["spans"])+1)
ax[0].semilogy(iterations, one_m2["spans"], marker="o")
ax[0].axhline(EPSILON, linestyle="--", color="gray", label="epsilon")
ax[0].set(xlabel="Iteration", ylabel="Span (log scale)", title="Convergence: max Delta - min Delta")
ax[0].legend()
ax[1].plot(iterations, one_m2["gains"], marker="o")
ax[1].axhline(2.215, linestyle="--", color="gray", label="Paper ~2.215")
ax[1].set(xlabel="Iteration", ylabel="Mean increment", title="Estimated average reward")
ax[1].legend()
plt.tight_layout(); plt.show()

display(pd.DataFrame({"iteration":iterations, "span":one_m2["spans"],
                      "estimated_gain":one_m2["gains"]}).tail(8))

### Optimal-order heatmap by inventory age

The horizontal axis represents **fresh inventory** and the vertical axis represents **old inventory**. Each cell displays the order quantity selected for that state. Orders are placed today and arrive **tomorrow**. This visualization highlights how optimal replenishment depends on inventory age, not merely total stock.


In [ ]:
Q = one_m2["Q"]
policy_map = one_m2["policy"].reshape(Q+1,Q+1)
fig, ax = plt.subplots(figsize=(7,5.7))
im = ax.imshow(policy_map, origin="lower", cmap="viridis", aspect="auto")
for old in range(Q+1):
    for fresh in range(Q+1):
        ax.text(fresh,old,str(policy_map[old,fresh]),ha="center",va="center",
                fontsize=8,color="white" if policy_map[old,fresh]<5 else "black")
ax.set(xlabel="Fresh stock",ylabel="Old stock", title="Optimal order quantity q*(old,fresh), M=2 Q=9")
fig.colorbar(im,ax=ax,label="Order quantity"); plt.tight_layout();plt.show()
print("State (0,0) ->",policy_map[0,0],"; (9,0) ->",policy_map[9,0],"; (0,9) ->",policy_map[0,9])

## 4. Experiment B — Single product (`M=3, Q=15`) [GPU]

With shelf life `M=3`, the state is `(oldest, middle, freshest)`. The space contains **4,096 states**, updated in parallel using the **same CUDA Bellman kernel** as Experiment A.

**Reference:** average daily profit approximately **2.4**. Store the result as `one_m3`.


In [ ]:
t0 = time.perf_counter()
states, values, policy, spans, gains = solve_single(M=3, Q=15)
one_m3 = dict(M=3, Q=15, states=states, values=values, policy=policy,
              spans=spans, gains=gains)
print(f"VI daily profit = {gains[-1]:.6f} (paper ~2.40)")
print(f"Iterations = {len(spans)}, final span = {spans[-1]:.7g}")
print(f"Optimal order at zero inventory = {policy[0]}")
print(f"Elapsed = {time.perf_counter()-t0:.2f}s")

plt.figure(figsize=(7,3.3))
plt.semilogy(np.arange(1,len(spans)+1),spans,marker="o")
plt.axhline(EPSILON,color="gray",linestyle="--")
plt.xlabel("Iteration");plt.ylabel("Span (log scale)");plt.title("Single Product M=3 convergence")
plt.show()

## 5. Two-product model: one-way substitution from B to A

Let \(D_A,D_B\sim\mathrm{Poisson}(5)\). When demand for product B exceeds available B inventory, each unserved B customer switches to product A independently with probability \(\gamma=0.5\), subject to A availability.

\[
U\mid(D_B-Y_B)^+=x\sim\mathrm{Binomial}(x,\gamma),\qquad
S_A=\min(Y_A,D_A+U),\qquad S_B=\min(Y_B,D_B).
\]

`joint[ya, yb, sa, sb]` stores the probability of sales \((s_a,s_b)\) conditional on available inventories \((y_a,y_b)\). The function `joint_sales_prob_fast()` **precomputes** these probabilities so that value iteration can reuse them efficiently.


In [ ]:
def joint_sales_prob_fast(max_a,max_b,mu_a=5.,mu_b=5.,gamma=.5):
    max_db=int(max(75,poisson.ppf(1-1e-14,mu_b)+5))
    pb=poisson.pmf(np.arange(max_db+1),mu_b)
    out=np.zeros((max_a+1,max_b+1,max_a+1,max_b+1))
    for ya in range(max_a+1):
        pa=capped_poisson(mu_a,ya)
        for yb in range(max_b+1):
            for sb in range(yb):out[ya,yb,:ya+1,sb]=pa*pb[sb]
            pu=np.zeros(max_db-yb+1)
            for db in range(yb,max_db+1):
                ex=db-yb
                pu[:ex+1]+=pb[db]*binom.pmf(np.arange(ex+1),ex,gamma)
            for u,p in enumerate(pu):
                if p<1e-16:continue
                if u>=ya:out[ya,yb,ya,yb]+=p
                else:
                    if ya-u>0:out[ya,yb,u:ya,yb]+=p*poisson.pmf(np.arange(ya-u),mu_a)
                    out[ya,yb,ya,yb]+=p*poisson.sf(ya-u-1,mu_a)
            mass=out[ya,yb].sum()
            if abs(mass-1)>1e-10:raise ValueError((ya,yb,mass))
            out[ya,yb]/=mass
    return out

In [ ]:
joint_demo = joint_sales_prob_fast(5, 5, MU_A, MU_B, GAMMA)
print("Sum of joint probabilities at (Ya,Yb)=(3,3):",joint_demo[3,3].sum())
fig,ax=plt.subplots(figsize=(5.5,4))
im=ax.imshow(joint_demo[3,3,:4,:4],origin="lower",cmap="magma",aspect="auto")
ax.set(xlabel="Sales B",ylabel="Sales A",title="P(Sa,Sb | Ya=3,Yb=3)")
fig.colorbar(im,ax=ax);plt.show()

## 6. Two-product Bellman update on the GPU

The state is `(a_old, a_fresh, b_old, b_fresh)` and the order is `(qa, qb)`. **B → A substitution** is captured by the joint capped-sales probability table `joint[ya,yb,sa,sb]`.

\[
V_{k+1}(I)=s_a\mathbb E[S_A\mid I]+s_b\mathbb E[S_B\mid I]
+\max_{q_a,q_b}\left[-c_a q_a-c_b q_b+\sum_{s_a,s_b}p(s_a,s_b\mid I)V_k(F(I,q_a,q_b,s_a,s_b))\right].
\]

The GPU update has **two kernels**:

- `two_action_values_gpu`: **one GPU thread per (state, order pair)**; computes the expected future value for that candidate order.
- `two_select_policy_gpu`: **one GPU thread per state**; chooses the best order pair, adds today's expected sales revenue, and computes the Bellman increment.

This retains the CPU baseline's transition logic, probability cutoff (`p < 1e-14`), state ordering, and tie-breaking. The demand table is created once on the CPU, then copied to the GPU.


In [ ]:
@cuda.jit
def two_action_values_gpu(joint, W, candidates, Qa, Qb, ca, cb):
    # One GPU thread evaluates one inventory state and one order pair.
    idx = cuda.grid(1)
    na, nb = Qa + 1, Qb + 1
    actions = na * nb
    states = na * na * nb * nb
    if idx >= states * actions:
        return

    j = idx // actions
    aidx = idx - j * actions
    qa = aidx // nb
    qb = aidx % nb

    bf = j % nb
    t = j // nb
    bo = t % nb
    t = t // nb
    af = t % na
    ao = t // na
    ya = ao + af
    yb = bo + bf

    future = 0.0
    for sa in range(ya + 1):
        ra = af - max(0, sa - ao)
        if ra < 0:
            ra = 0
        for sb in range(yb + 1):
            p = joint[ya, yb, sa, sb]
            if p < 1e-14:
                continue
            rb = bf - max(0, sb - bo)
            if rb < 0:
                rb = 0
            k = ((ra * na + qa) * nb + rb) * nb + qb
            future += p * W[k]
    candidates[idx] = future - ca * qa - cb * qb


@cuda.jit
def two_select_policy_gpu(candidates, expected_a, expected_b,
                          W, V, policy_a, policy_b, increments,
                          Qa, Qb, price_a, price_b):
    # One GPU thread selects the best action for one inventory state.
    j = cuda.grid(1)
    na, nb = Qa + 1, Qb + 1
    states = na * na * nb * nb
    if j >= states:
        return

    bf = j % nb
    t = j // nb
    bo = t % nb
    t = t // nb
    af = t % na
    ao = t // na
    ya = ao + af
    yb = bo + bf

    best, best_qa, best_qb = -1e100, 0, 0
    base = j * na * nb
    for qa in range(na):
        for qb in range(nb):
            value = candidates[base + qa * nb + qb]
            if value > best:
                best, best_qa, best_qb = value, qa, qb

    v = price_a * expected_a[ya, yb] + price_b * expected_b[ya, yb] + best
    V[j] = v
    increments[j] = v - W[j]
    policy_a[j] = best_qa
    policy_b[j] = best_qb


def solve_two_gpu(joint, Qa, Qb, s_a=1., s_b=1., c_a=.5, c_b=.5,
                  epsilon=EPSILON, max_iter=100):
    # Expected sales depend only on current inventory (not on today's orders).
    ea = np.zeros((2*Qa+1, 2*Qb+1), dtype=np.float64)
    eb = np.zeros_like(ea)
    for ya in range(2*Qa+1):
        for yb in range(2*Qb+1):
            p = joint[ya, yb, :ya+1, :yb+1]
            ea[ya, yb] = (p * np.arange(ya+1)[:, None]).sum()
            eb[ya, yb] = (p * np.arange(yb+1)[None, :]).sum()

    # Copy fixed probability tables once; keep W and V on the GPU.
    d_joint = cuda.to_device(np.ascontiguousarray(joint, dtype=np.float64))
    d_ea, d_eb = cuda.to_device(ea), cuda.to_device(eb)
    na, nb = Qa + 1, Qb + 1
    shape = (na, na, nb, nb)
    n_states = na * na * nb * nb
    n_actions = na * nb
    d_w = cuda.to_device(np.zeros(n_states, dtype=np.float64))
    d_v = cuda.device_array(n_states, dtype=np.float64)
    d_candidates = cuda.device_array(n_states * n_actions, dtype=np.float64)
    d_pa = cuda.device_array(n_states, dtype=np.int32)
    d_pb = cuda.device_array(n_states, dtype=np.int32)
    d_inc = cuda.device_array(n_states, dtype=np.float64)
    threads = 128
    blocks_actions = (n_states * n_actions + threads - 1) // threads
    blocks_states = (n_states + threads - 1) // threads
    spans, gains = [], []

    cuda.synchronize()
    t0 = time.perf_counter()
    for iteration in range(max_iter):
        two_action_values_gpu[blocks_actions, threads](
            d_joint, d_w, d_candidates, Qa, Qb, c_a, c_b)
        two_select_policy_gpu[blocks_states, threads](
            d_candidates, d_ea, d_eb, d_w, d_v, d_pa, d_pb, d_inc,
            Qa, Qb, s_a, s_b)
        delta = d_inc.copy_to_host()  # Synchronizes both GPU kernels.
        span = float(delta.max() - delta.min())
        spans.append(span)
        gains.append(float(delta.mean()))
        if span < epsilon:
            break
        d_w, d_v = d_v, d_w
    cuda.synchronize()
    runtime = time.perf_counter() - t0
    print(f"GPU Bellman runtime (excluding probability preparation) = {runtime:.3f}s")
    return (d_v.copy_to_host().reshape(shape),
            d_pa.copy_to_host().reshape(shape),
            d_pb.copy_to_host().reshape(shape),
            np.array(spans), np.array(gains))


## 7. Experiment C — Two products (`Qa=Qb=10`) [GPU]

This experiment evaluates **14,641 states** and **121 order pairs per state**, with a GPU thread assigned to each state–action combination.

**CPU reference:** profit ≈ **4.503204**, 13 iterations, zero-stock order `(8, 7)` (as observed in the prior CPU run). The published **5.509** figure is not matched by the CPU implementation; do not treat that number as a GPU target until the discrepancy is resolved.

**Performance note:** GPU kernel compilation and host↔device transfers affect wall time. Compare steady-state GPU and CPU times only under matched conditions and with compilation excluded.


In [ ]:
QA = QB = 10   # For a quick GPU test, try QA = QB = 3 first.
t0 = time.perf_counter()
joint = joint_sales_prob_fast(2*QA, 2*QB, MU_A, MU_B, GAMMA)
V2, pa2, pb2, spans2, gains2 = solve_two_gpu(
    joint, QA, QB, PRICE_A, PRICE_B, COST_A, COST_B, EPSILON, 100)
two_result = dict(Qa=QA, Qb=QB, values=V2, policy_a=pa2, policy_b=pb2,
                  spans=spans2, gains=gains2)
print(f"GPU VI daily profit = {gains2[-1]:.6f} (CPU reference ~4.503204)")
print(f"Iterations = {len(spans2)}, final span = {spans2[-1]:.7g}")
print("Optimal order at zero inventory =", (int(pa2[0,0,0,0]), int(pb2[0,0,0,0])))
print(f"Wall time (including preparation/compilation) = {time.perf_counter()-t0:.2f}s")


### Two-product convergence and policy slices

The full policy depends on four inventory coordinates, so a single two-dimensional plot cannot represent every state. Instead, the plots show **two-dimensional slices**: the optimal A order when B inventory is zero, and the optimal B order when A inventory is zero. A separate panel plots the two-product convergence span.


In [ ]:
fig,axs=plt.subplots(1,3,figsize=(15,4))
it=np.arange(1,len(two_result["spans"])+1)
axs[0].semilogy(it,two_result["spans"],marker="o")
axs[0].axhline(EPSILON,linestyle="--",color="gray")
axs[0].set(xlabel="Iteration",ylabel="Span",title="Two product convergence")
im1=axs[1].imshow(two_result["policy_a"][:,:,0,0],origin="lower",cmap="viridis")
axs[1].set(xlabel="A fresh",ylabel="A old",title="A orders (B stock = 0)")
fig.colorbar(im1,ax=axs[1],fraction=.046)
im2=axs[2].imshow(two_result["policy_b"][0,0,:,:],origin="lower",cmap="viridis")
axs[2].set(xlabel="B fresh",ylabel="B old",title="B orders (A stock = 0)")
fig.colorbar(im2,ax=axs[2],fraction=.046)
plt.tight_layout();plt.show()

## 8. Monte Carlo policy evaluation (CPU)

Value iteration runs on the **GPU** and analytically sums demand probabilities. Simulation draws **random daily Poisson/Binomial demand** and evolves inventory **one day at a time**.

The simulation runs for **400,000 days** by default. It remains on the CPU because inventory updates are sequential in time. The GPU-generated optimal policies (`one_m2`, `two_result`) are copied back to the CPU and can be evaluated without changing the original simulation functions.


In [ ]:
def simulate_one(policy, M, Q, mu=5., s=1., c=.5, periods=400_000, seed=123):
    rng = np.random.default_rng(seed)
    I = [0] * M
    profits = np.empty(periods)
    waste = orders = 0
    demand = rng.poisson(mu, size=periods)
    running = np.empty(periods // 1000)
    for t, d in enumerate(demand):
        j = 0
        for x in I:
            j = j * (Q+1) + x
        q = int(policy[j])
        sales = min(sum(I), int(d))
        profits[t] = s*sales - c*q
        waste += max(0, I[0]-d)
        orders += q
        remaining = int(d)
        for r in range(M):
            take = min(I[r], remaining)
            I[r] -= take
            remaining -= take
        I = I[1:] + [q]
        if (t+1) % 1000 == 0:
            running[t // 1000] = profits[:t+1].mean()
    return dict(profit=float(profits.mean()), waste_pct=100*waste/orders,
                running_profit=running)


def simulate_two(result, periods=400_000, seed=123, mu_a=5., mu_b=5.,
                 gamma=.5, s_a=1., s_b=1., c_a=.5, c_b=.5):
    rng = np.random.default_rng(seed)
    pa, pb = result['policy_a'], result['policy_b']
    ao=af=bo=bf=0
    sum_profit=0.
    waste_a=waste_b=orders_a=orders_b=0
    running = np.empty(periods // 1000)
    for t in range(periods):
        qa=int(pa[ao,af,bo,bf]); qb=int(pb[ao,af,bo,bf])
        da=rng.poisson(mu_a); db=rng.poisson(mu_b)
        sub=rng.binomial(max(0, db-bo-bf), gamma)
        sold_a=min(ao+af, da+sub); sold_b=min(bo+bf, db)
        sum_profit += s_a*sold_a+s_b*sold_b-c_a*qa-c_b*qb
        waste_a += max(0, ao-sold_a); waste_b += max(0, bo-sold_b)
        orders_a += qa; orders_b += qb
        af=max(0, af-max(0, sold_a-ao)); bf=max(0, bf-max(0, sold_b-bo))
        ao,af,bo,bf=af,qa,bf,qb
        if (t+1) % 1000 == 0:
            running[t // 1000] = sum_profit/(t+1)
    return dict(profit=sum_profit/periods, waste_a_pct=100*waste_a/orders_a,
                waste_b_pct=100*waste_b/orders_b, running_profit=running)


def simulate_base_stock_two(Sa=13,Sb=12,periods=400_000,seed=123,
                            mu_a=5.,mu_b=5.,gamma=.5,
                            s_a=1.,s_b=1.,c_a=.5,c_b=.5):
    rng=np.random.default_rng(seed)
    ao=af=bo=bf=0
    profit=0.; waste_a=waste_b=orders_a=orders_b=0
    running = np.empty(periods // 1000)
    for t in range(periods):
        qa=max(0, Sa-ao-af+max(0,ao-mu_a))
        qb=max(0, Sb-bo-bf+max(0,bo-mu_b))
        da=rng.poisson(mu_a);db=rng.poisson(mu_b)
        sub=rng.binomial(max(0,db-bo-bf),gamma)
        sa=min(ao+af,da+sub);sb=min(bo+bf,db)
        profit += s_a*sa+s_b*sb-c_a*qa-c_b*qb
        waste_a += max(0,ao-sa);waste_b += max(0,bo-sb)
        orders_a += qa;orders_b += qb
        next_ao=max(0,af-max(0,sa-ao))
        next_bo=max(0,bf-max(0,sb-bo))
        ao,af,bo,bf=next_ao,qa,next_bo,qb
        if (t+1) % 1000 == 0:
            running[t // 1000] = profit/(t+1)
    return dict(profit=profit/periods,waste_a_pct=100*waste_a/orders_a,
                waste_b_pct=100*waste_b/orders_b,running_profit=running)

### Experiment D — Single-product simulation

Run Experiment A first to construct `one_m2`, then execute this cell. Compare the **value-iteration average-reward estimate** with the **realized average profit** over the simulated demand path.


In [ ]:
one_sim = simulate_one(one_m2["policy"], M=2, Q=9, periods=400_000, seed=SEED)
print("VI estimate:", f'{one_m2["gains"][-1]:.6f}')
print("Simulation average:", f'{one_sim["profit"]:.6f}')
print("Waste (% of ordered units):", f'{one_sim["waste_pct"]:.3f}%')
plt.figure(figsize=(7,3.5))
plt.plot(np.arange(1,len(one_sim["running_profit"])+1)*1000,one_sim["running_profit"],label="Simulation running mean")
plt.axhline(one_m2["gains"][-1],ls="--",c="gray",label="Value Iteration")
plt.xlabel("Simulated days");plt.ylabel("Average daily profit");plt.legend();plt.show()

### Experiment E — Two-product simulation and base-stock benchmark

Run Experiment C first to construct `two_result`. This experiment evaluates the optimized ordering policy and a waste-conscious base-stock benchmark.

Although both policy simulations use the same random seed, their random-number consumption may differ, so the realized demand paths are **not necessarily identical**. A more rigorous paired comparison would use the same pre-generated exogenous demand sequence for both policies.

Previously observed terminal results (400,000 simulated days): **VI simulation profit = 4.500659**, **base-stock profit = 4.469071**. Re-run these cells to check reproducibility in the notebook environment.


In [ ]:
t0=time.perf_counter()
vi_sim = simulate_two(two_result, periods=400_000, seed=SEED,
                      mu_a=MU_A,mu_b=MU_B,gamma=GAMMA,
                      s_a=PRICE_A,s_b=PRICE_B,c_a=COST_A,c_b=COST_B)
base_sim = simulate_base_stock_two(periods=400_000, seed=SEED,
                      mu_a=MU_A,mu_b=MU_B,gamma=GAMMA,
                      s_a=PRICE_A,s_b=PRICE_B,c_a=COST_A,c_b=COST_B)
summary = pd.DataFrame([
    {"Method":"Value Iteration (model)","Daily profit":two_result["gains"][-1],"Waste A (%)":np.nan,"Waste B (%)":np.nan},
    {"Method":"VI policy (simulation)","Daily profit":vi_sim["profit"],"Waste A (%)":vi_sim["waste_a_pct"],"Waste B (%)":vi_sim["waste_b_pct"]},
    {"Method":"Base-stock (simulation)","Daily profit":base_sim["profit"],"Waste A (%)":base_sim["waste_a_pct"],"Waste B (%)":base_sim["waste_b_pct"]},
])
display(summary)
print(f"Simulation time: {time.perf_counter()-t0:.2f}s")

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
methods=["VI policy","Base-stock"]
profits=[vi_sim["profit"],base_sim["profit"]]
axs[0].bar(methods,profits)
axs[0].set(ylabel="Average daily profit",title="Policy performance (simulation)")
for x,y in enumerate(profits):axs[0].text(x,y+.01,f"{y:.4f}",ha="center")
xs=np.arange(2);width=.34
axs[1].bar(xs-width/2,[vi_sim["waste_a_pct"],base_sim["waste_a_pct"]],width,label="Product A")
axs[1].bar(xs+width/2,[vi_sim["waste_b_pct"],base_sim["waste_b_pct"]],width,label="Product B")
axs[1].set_xticks(xs,methods)
axs[1].set(ylabel="Waste (% of orders)",title="Expired stock / ordered units")
axs[1].legend();plt.tight_layout();plt.show()

## 9. Interpretation, validation, and next steps

### How to interpret the results
- **CUDA computation:** All three value-iteration experiments use GPU Bellman-update kernels. Demand-probability preprocessing, convergence checks, figures, and simulation remain on the CPU.
- **Convergence:** `span < 1e-4` measures the spread of Bellman increments across states, not whether `V` tends to zero.
- **Policy:** `policy` or `(policy_a, policy_b)` stores the maximizing order decision for each state.
- **Average reward:** `gains[-1]` estimates long-run expected daily profit.
- **Simulation:** Random demand is generated on the CPU over 400,000 days, preserving inventory dependencies.
- **Two-product discrepancy:** The existing CPU baseline yields about **4.503**, rather than the paper's **5.509**; GPU and CPU should agree with each other before investigating the published figure.

### Validation checklist
1. **Experiment A** should give approximately `2.215149`, with zero-stock order `7`.
2. **Experiment B** should give approximately `2.398490`, with zero-stock order `9`.
3. **Experiment C** should give approximately `4.503204`, with zero-stock order `(8,7)`.
4. For the same parameters, compare the CPU reference and GPU **policy arrays**, average reward, and final span (rounding and GPU reduction order can cause small differences).
5. Compare **steady-state GPU time**, excluding initial CUDA JIT compilation, against a warmed-up CPU baseline. Use `cuda.synchronize()` around GPU timing.

### Important limitations
- CUDA requires a compatible NVIDIA GPU; **Apple Silicon does not execute NVIDIA CUDA kernels**.
- Running on GPU does not guarantee a speedup for small state spaces; kernel launch overhead and memory access matter.
- The two-product code uses a thread per **state–action pair**, rather than the paper's exact GPU block/thread mapping per order/event. The Bellman model is retained, but benchmark speedups are **not directly comparable** with the original 2019 hardware results.
- The `numba` CUDA simulator can test small cases, but its runtimes are **not GPU benchmarks**.
